In [1]:
import os

In [2]:
%pwd

'd:\\DS_Project\\data_science_project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\DS_Project\\data_science_project'

In [7]:
import pandas as pd
data = pd.read_csv("artifacts/data_ingestion/data.csv", sep = ";")
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4898 entries, 0 to 4897
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         4898 non-null   float64
 1   volatile acidity      4898 non-null   float64
 2   citric acid           4898 non-null   float64
 3   residual sugar        4898 non-null   float64
 4   chlorides             4898 non-null   float64
 5   free sulfur dioxide   4898 non-null   float64
 6   total sulfur dioxide  4898 non-null   float64
 7   density               4898 non-null   float64
 8   pH                    4898 non-null   float64
 9   sulphates             4898 non-null   float64
 10  alcohol               4898 non-null   float64
 11  quality               4898 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 459.3 KB


In [9]:
data.isna().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [10]:
data.shape

(4898, 12)

In [42]:
from dataclasses import dataclass 
from pathlib import Path

@dataclass
class DataValidationConfig:
    root_dir: Path
    store_data_dir: Path
    STATUS_FILE: str
    all_schema: dict

In [43]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.schema = read_yaml(schema_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:

        config = self.config.data_validation

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            store_data_dir=config.store_data_dir,
            all_schema=self.schema.COLUMNS,
        )

        return data_validation_config

In [45]:
import os
from src.datascience import logger

In [51]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_columns(self) -> bool:
        try:
            validation_status = True

            data = pd.read_csv(self.config.store_data_dir, sep = ";")

            all_cols = list(data.columns)
            all_schema = self.config.all_schema.keys()

            for col in all_cols:
                if col not in all_schema:
                    validation_status = False
                    print(f"Column '{col}' not found in schema.")
                    break

            with open(self.config.STATUS_FILE, "w") as f:
                f.write(f"Validation status: {validation_status}")

            return validation_status

        except Exception as e:
            raise e

In [52]:
try:
    config = ConfigurationManager()

    data_validation_config = config.get_data_validation_config()

    data_validation = DataValidation(config=data_validation_config)

    validation_status = data_validation.validate_all_columns()

    print(validation_status)

except Exception as e:
    raise e

[2026-07-22 15:58:30,519: INFO: common: yaml file: config\config.yaml is loaded successfully]
[2026-07-22 15:58:30,529: INFO: common: yaml file: schema.yaml is loaded successfully]
[2026-07-22 15:58:30,535: INFO: common: Created directory at: artifacts]
[2026-07-22 15:58:30,539: INFO: common: Created directory at: artifacts/data_validation]
True
